# Token Label Explorer

Load augmented LiveCodeBench JSONL files and explore token-level labels alongside fixed programs.

In [ ]:
import html
import json
import re
from pathlib import Path
from difflib import SequenceMatcher

import ipywidgets as widgets
from IPython.display import display, HTML

try:
    from transformers import AutoTokenizer
except ImportError:
    AutoTokenizer = None

NO_FIX_FOUND_SENTINEL = "<NO_FIX_FOUND>"
NO_PROGRAM_SENTINEL = "<NO_PROGRAM>"
CORRECT_PROGRAM_SENTINEL = "<CORRECT_PROGRAM>"


def read_jsonl(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def candidate_count(record: dict) -> int:
    return max(
        len(record.get("program", []) or []),
        len(record.get("token_labels", []) or []),
        len(record.get("fixed_program", []) or []),
        0,
    )


def _build_force_true_mask_for_text(tokenizer, text: str, token_ids):
    if tokenizer is None or not text or not token_ids:
        return [False for _ in token_ids]
    comment_spans = [(m.start(), m.end()) for m in re.finditer(r"#.*", text)]
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    offsets = encoded.get("offset_mapping") or []
    if len(offsets) != len(token_ids):
        return [False for _ in token_ids]
    force_true = []
    for start, end in offsets:
        if start is None or end is None or start >= end:
            force_true.append(False)
            continue
        segment = text[start:end]
        is_whitespace = bool(segment) and all(ch.isspace() for ch in segment)
        is_comment = any(start < span_end and end > span_start for span_start, span_end in comment_spans)
        force_true.append(is_whitespace or is_comment)
    return force_true


def _compute_code_only_equal_masks(
    buggy_tokens,
    fixed_tokens,
    buggy_force_true_mask=None,
    fixed_force_true_mask=None,
):
    buggy_equal_mask = [False for _ in buggy_tokens]
    fixed_equal_mask = [False for _ in fixed_tokens]
    if not buggy_tokens or not fixed_tokens:
        return buggy_equal_mask, fixed_equal_mask, False

    matcher = SequenceMatcher(a=buggy_tokens, b=fixed_tokens, autojunk=False)
    has_equal_match = any(tag == "equal" for tag, *_ in matcher.get_opcodes())

    buggy_mask = buggy_force_true_mask or [False for _ in buggy_tokens]
    fixed_mask = fixed_force_true_mask or [False for _ in fixed_tokens]
    if len(buggy_mask) != len(buggy_tokens):
        buggy_mask = [False for _ in buggy_tokens]
    if len(fixed_mask) != len(fixed_tokens):
        fixed_mask = [False for _ in fixed_tokens]

    buggy_code_indices = [idx for idx, is_masked in enumerate(buggy_mask) if not is_masked]
    fixed_code_indices = [idx for idx, is_masked in enumerate(fixed_mask) if not is_masked]
    if not buggy_code_indices or not fixed_code_indices:
        return buggy_equal_mask, fixed_equal_mask, has_equal_match

    buggy_code_tokens = [buggy_tokens[idx] for idx in buggy_code_indices]
    fixed_code_tokens = [fixed_tokens[idx] for idx in fixed_code_indices]
    code_matcher = SequenceMatcher(a=buggy_code_tokens, b=fixed_code_tokens, autojunk=False)
    for tag, a0, a1, b0, b1 in code_matcher.get_opcodes():
        if tag == "equal":
            for code_idx in range(a0, a1):
                buggy_equal_mask[buggy_code_indices[code_idx]] = True
            for code_idx in range(b0, b1):
                fixed_equal_mask[fixed_code_indices[code_idx]] = True
    return buggy_equal_mask, fixed_equal_mask, has_equal_match


def highlight_fixed_program(program_text: str, fixed_text: str, tokenizer) -> str:
    if not fixed_text:
        return ""
    if tokenizer is None:
        return html.escape(fixed_text).replace("\n", "<br>")

    program_encoded = tokenizer(
        program_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    fixed_encoded = tokenizer(
        fixed_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    program_ids = program_encoded.get("input_ids") or []
    fixed_ids = fixed_encoded.get("input_ids") or []
    fixed_offsets = fixed_encoded.get("offset_mapping") or []

    if not program_ids or not fixed_ids or len(fixed_offsets) != len(fixed_ids):
        return html.escape(fixed_text).replace("\n", "<br>")

    program_force_true_mask = _build_force_true_mask_for_text(tokenizer, program_text, program_ids)
    fixed_force_true_mask = _build_force_true_mask_for_text(tokenizer, fixed_text, fixed_ids)

    _, fixed_equal_mask, _ = _compute_code_only_equal_masks(
        program_ids,
        fixed_ids,
        buggy_force_true_mask=program_force_true_mask,
        fixed_force_true_mask=fixed_force_true_mask,
    )

    rendered = []
    cursor = 0
    for idx, (start, end) in enumerate(fixed_offsets):
        if start is None or end is None or start >= end:
            continue
        if start > cursor:
            rendered.append(html.escape(fixed_text[cursor:start]).replace("\n", "<br>"))
        segment = html.escape(fixed_text[start:end]).replace("\n", "<br>")
        should_highlight = fixed_equal_mask[idx] or (idx < len(fixed_force_true_mask) and fixed_force_true_mask[idx])
        if should_highlight:
            rendered.append(f"<span class='match-token'>{segment}</span>")
        else:
            rendered.append(segment)
        cursor = end

    if cursor < len(fixed_text):
        rendered.append(html.escape(fixed_text[cursor:]).replace("\n", "<br>"))
    return "".join(rendered)


def token_label_fields(token_label):
    if isinstance(token_label, dict):
        return (
            token_label.get("token_id"),
            token_label.get("token_text"),
            token_label.get("label"),
        )
    if isinstance(token_label, (list, tuple)) and len(token_label) >= 3:
        return token_label[0], token_label[1], token_label[2]
    return None, None, None


def render_program_with_labels_by_tokenizer(program_text: str, token_labels, tokenizer):
    if not program_text or not token_labels or tokenizer is None:
        return None
    encoded = tokenizer(
        program_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    offsets = encoded.get("offset_mapping")
    program_ids = encoded.get("input_ids")
    if not offsets or not program_ids or len(offsets) != len(program_ids):
        return None
    label_ids = []
    label_values = []
    for item in token_labels:
        token_id, _token_text, label_value = token_label_fields(item)
        label_ids.append(token_id)
        label_values.append(label_value)
    matcher = SequenceMatcher(a=label_ids, b=program_ids, autojunk=False)
    program_label_map = [None] * len(program_ids)
    for tag, a0, a1, b0, b1 in matcher.get_opcodes():
        if tag == "equal":
            for i in range(b0, b1):
                program_label_map[i] = label_values[a0 + (i - b0)]
    rendered = []
    cursor = 0
    for (start, end), label in zip(offsets, program_label_map):
        if start is None or end is None or start == end:
            continue
        if start > cursor:
            rendered.append(html.escape(program_text[cursor:start]).replace("\n", "<br>"))
        segment = html.escape(program_text[start:end]).replace("\n", "<br>")
        if label is True:
            rendered.append(f"<span class='label-true'>{segment}</span>")
        elif label is False:
            rendered.append(f"<span class='label-false'>{segment}</span>")
        else:
            rendered.append(segment)
        cursor = end
    if cursor < len(program_text):
        rendered.append(html.escape(program_text[cursor:]).replace("\n", "<br>"))
    return "".join(rendered)


def render_program_with_labels_fallback(token_labels) -> str:
    if not token_labels:
        return ""
    rendered = []
    for item in token_labels:
        token_id, token_text, label = token_label_fields(item)
        token_text = token_text if token_text is not None else str(token_id)
        token_text = token_text.replace("Ġ", " ").replace("▁", " ")
        safe = html.escape(token_text).replace("\n", "<br>")
        css_class = "label-true" if label else "label-false"
        rendered.append(f"<span class='{css_class}'>{safe}</span>")
    return "".join(rendered)


STYLE = """
<style>
  .container { display: flex; gap: 16px; align-items: flex-start; }
  .panel { width: 50%; border: 1px solid #ddd; border-radius: 8px; padding: 12px; }
  .panel h3 { margin-top: 0; }
  .label-true { background: #d1f5d3; border-radius: 3px; padding: 0 2px; }
  .label-false { background: #f8d7da; border-radius: 3px; padding: 0 2px; }
  .mono { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace; white-space: pre-wrap; }
  .meta { color: #666; font-size: 0.9em; margin-bottom: 8px; }
  .prompt-details { margin: 8px 0 12px; }
  .prompt-details summary { cursor: pointer; font-weight: 600; }
  .prompt-box { margin-top: 8px; border: 1px solid #eee; border-radius: 6px; padding: 8px; background: #fafafa; }
</style>
"""

path_widget = widgets.Text(
    value="/data/livecodebench/augmentation/livecodebench_augmented_Qwen_Qwen3-Coder-30B-A3B-Instruct_full.jsonl",
    description="JSONL path:",
    layout=widgets.Layout(width="70%"),
)
tokenizer_widget = widgets.Text(
    value="Qwen/Qwen3-Coder-30B-A3B-Instruct",
    description="Tokenizer:",
    layout=widgets.Layout(width="70%"),
)
load_tokenizer_button = widgets.Button(description="Load tokenizer")
load_button = widgets.Button(description="Load JSONL", button_style="primary")
row_dropdown = widgets.Dropdown(options=[], description="Row:", layout=widgets.Layout(width="50%"))
candidate_dropdown = widgets.Dropdown(options=[], description="Candidate:", layout=widgets.Layout(width="30%"))
status = widgets.HTML(value="Ready.")
output = widgets.Output()

records = []
tokenizer = None


def load_tokenizer(_=None):
    global tokenizer
    if AutoTokenizer is None:
        status.value = "<span style='color:red'>transformers not installed.</span>"
        return
    name = tokenizer_widget.value.strip()
    if not name:
        status.value = "<span style='color:red'>Tokenizer name required.</span>"
        return
    status.value = f"Loading tokenizer: {html.escape(name)}"
    try:
        tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        status.value = f"Tokenizer loaded: {html.escape(name)}"
        render()
    except Exception as exc:
        tokenizer = None
        status.value = f"<span style='color:red'>Failed to load tokenizer: {html.escape(str(exc))}</span>"


def load_records(_=None):
    global records
    output.clear_output()
    path = Path(path_widget.value).expanduser()
    if not path.exists():
        status.value = f"<span style='color:red'>File not found: {path}</span>"
        records = []
        row_dropdown.options = []
        candidate_dropdown.options = []
        return
    records = read_jsonl(path)
    row_options = [(f"Row {i}", i) for i in range(len(records))]
    row_dropdown.options = row_options
    row_dropdown.value = row_options[0][1] if row_options else None
    status.value = f"Loaded {len(records)} rows from {path}."
    update_candidates()


def update_candidates(*_):
    if not records or row_dropdown.value is None:
        candidate_dropdown.options = []
        return
    record = records[row_dropdown.value]
    count = candidate_count(record)
    candidate_dropdown.options = [(str(i), i) for i in range(count)]
    candidate_dropdown.value = 0 if count else None
    render()


def render(*_):
    output.clear_output()
    if not records or row_dropdown.value is None or candidate_dropdown.value is None:
        return
    record = records[row_dropdown.value]
    idx = candidate_dropdown.value
    program = (record.get("program", []) or [])
    program_text = program[idx] if idx < len(program) else ""
    fixed_programs = (record.get("fixed_program", []) or [])
    fixed_text = fixed_programs[idx] if idx < len(fixed_programs) else ""
    token_labels_all = (record.get("token_labels", []) or [])
    token_labels = token_labels_all[idx] if idx < len(token_labels_all) else []
    line_filters = (record.get("line_filter", []) or [])
    line_filter_value = line_filters[idx] if idx < len(line_filters) else False
    no_correct_line_filters = (record.get("no_correct_line_filter", []) or [])
    no_correct_line_filter_value = no_correct_line_filters[idx] if idx < len(no_correct_line_filters) else False

    header = (
        f"<div class='meta'>"
        f"Problem ID: <b>{html.escape(str(record.get('id', 'n/a')))}</b> | "
        f"Row: {row_dropdown.value} | Candidate: {idx}"
        f"</div>"
    )

    prompt_text = (
        record.get("prompt")
    )
    prompt_html = ""
    if prompt_text:
        prompt_body = html.escape(str(prompt_text)).replace("\n", "<br>")
        prompt_html = (
            "<details class='prompt-details'>"
            "<summary>Show prompt</summary>"
            f"<div class='mono prompt-box'>{prompt_body}</div>"
            "</details>"
        )

    line_filter_html = (
        f"<div class='meta'>Line filter: <b>{html.escape(str(line_filter_value))}</b> | "
        f"No correct line filter: <b>{html.escape(str(no_correct_line_filter_value))}</b></div>"
    )

    program_html = render_program_with_labels_by_tokenizer(program_text, token_labels, tokenizer)
    if program_html is None:
        program_html = render_program_with_labels_fallback(token_labels) or html.escape(program_text).replace("\n", "<br>")
    fixed_html = ""
    fixed_note = ""

    if fixed_text == NO_PROGRAM_SENTINEL:
        fixed_note = "No program available for this candidate."
    elif fixed_text == NO_FIX_FOUND_SENTINEL:
        fixed_note = "No fix passed the tests."
    elif fixed_text == CORRECT_PROGRAM_SENTINEL:
        fixed_note = "Program already correct."
        fixed_html = html.escape(program_text).replace("\n", "<br>")
    else:
        fixed_html = html.escape(fixed_text).replace("\n", "<br>")

    if fixed_note:
        fixed_html = f"<div class='meta'>{fixed_note}</div>" + fixed_html

    panel_html = (
        f"<div class='container'>"
        f"<div class='panel'><h3>Program (token labels)</h3><div class='mono'>{program_html}</div></div>"
        f"<div class='panel'><h3>Fixed program</h3><div class='mono'>{fixed_html}</div></div>"
        f"</div>"
    )

    with output:
        display(HTML(STYLE + header + prompt_html + line_filter_html + panel_html))


load_tokenizer_button.on_click(load_tokenizer)
load_button.on_click(load_records)
row_dropdown.observe(lambda change: update_candidates() if change.get("name") == "value" else None, names="value")
candidate_dropdown.observe(lambda change: render() if change.get("name") == "value" else None, names="value")

controls = widgets.VBox(
    [
        widgets.HBox([path_widget, load_button]),
        widgets.HBox([tokenizer_widget, load_tokenizer_button]),
        widgets.HBox([row_dropdown, candidate_dropdown]),
        status,
    ],
    layout=widgets.Layout(width="100%"),
)

display(controls, output)
